# Technical Analysis Agent — Module B
## FXAlphaLab: Multi-Agent FX Trading Framework

---

### 1.1 The Meese-Rogoff Problem & Why ML Changes the Equation

The dominant reference point for FX forecasting difficulty is **Meese & Rogoff (1983)**, who demonstrated
that structural macroeconomic models (monetary, portfolio-balance) could not out-predict a naïve random
walk out-of-sample, even with ex-post realised values of predictors [CITE: Meese, R. & Rogoff, K. (1983).
*Empirical Exchange Rate Models of the Seventies: Do They Fit Out of Sample?* Journal of International
Economics, 14(1–2), 3–24].

This notebook treats the random walk as the **Alpha Hurdle**: any model that cannot beat it in
directional accuracy has zero practical value, regardless of in-sample fit.

---
### 1.1.1 Dataset Description

This study uses OHLCV data for four major currency pairs — EUR/USD, GBP/USD, USD/JPY and
USD/CHF — across three timeframes (D1, H4, H1) spanning January 2021 to December 2025.
This five-year window captures some of the most structurally significant periods in recent
FX market history: COVID-19 recovery and dollar weakness (2021), the Ukraine war and
commodity shock (2022), the most aggressive Fed tightening cycle in four decades (2022–2023),
and the subsequent regime normalization (2024–2025). Far from being a limitation, this period
provides an exceptionally rich testing ground — models trained and validated here must
demonstrate robustness across multiple distinct market regimes, high volatility episodes,
and structural breaks. Critically, current conditions in 2025 suggest this volatility is
structural rather than transitional: ongoing US trade policy uncertainty, geopolitical
fragmentation, and diverging central bank cycles across the Fed, ECB and BoJ continue to
generate the kind of rapid regime shifts that static models cannot handle. This directly
motivates the HMM regime filter (Section 1.2, [10]) and the walk-forward validation
protocol (Section 1.2, [36]) central to Module B's architecture, and positions Module B's
regime-aware agent as not merely an academic exercise but a response to the current
reality of FX markets.

## 1.2 Why Deep Learning & Why Now

Meese & Rogoff's (1983) critique targeted structural macroeconomic models — it says nothing
about technical price-based approaches. Recent benchmark evidence supports LSTM as a reliable
architecture for FX trend forecasting, and signal decomposition as an effective preprocessing
step for deep learning on currency data:

> **[1]** Head-to-head benchmark of LSTM vs. CNN-Transformer on four major currency pairs
> (AUD/USD, CAD/CHF, EUR/GBP, GBP/USD) over a 20-year dataset, evaluated on MSE, RMSE,
> RME and MAPE: LSTM generally outperformed the hybrid CNN-Transformer across most pairs.
> The CNN-Transformer only outperformed LSTM on GBP/USD, with the authors concluding it
> works better on less noisy, less volatile data. This confirms LSTM as the reliable
> baseline for Module B, and justifies caution when applying Transformer-based architectures
> to volatile pairs like GBP/USD and USD/JPY.
> [CITE: *Transformer and LSTM Models for Forex Market Forecasting*, Research Bank NZ,
> https://www.researchbank.ac.nz/bitstreams/c6eb710f-7de2-43fc-826f-18900d9e19c3/download]

> **[2]** Lim, B., Arık, S.Ö., Loeff, N., & Pfister, T. (2021). *Temporal Fusion Transformers for
> Interpretable Multi-horizon Time Series Forecasting.* International Journal of Forecasting, 37(4),
> 1748–1764. https://arxiv.org/abs/1912.09363
> TFT is an attention-based architecture combining multi-horizon forecasting with
> interpretability. It uses recurrent layers for local temporal processing, self-attention layers
> for long-term dependencies, specialized components for feature selection, and gating layers to
> suppress irrelevant inputs. Crucially for Module B, TFT handles the complex mix of static
> covariates, known future inputs, and past-only exogenous series present in FX data — without
> requiring prior assumptions about how they interact.

> **[3]** Tang, X. & Xie, Y. (2025). *Exchange Rate Forecasting: A Deep Learning Framework Combining
> Adaptive Signal Decomposition and Dynamic Weight Optimization.* School of Business, Jiangnan University.
> Published 22 August 2025.
> Using OCEEMDAN (an optimized variant of CEEMDAN) as a preprocessing decomposition step before
> Bi-LSTM, GRU and FNN models on EUR/USD, GBP/USD and USD/JPY, the study demonstrates that
> decomposing non-stationary exchange rate signals into components prior to model input significantly
> improves forecasting accuracy and robustness. This motivates our application of CEEMDAN decomposition
> as a preprocessing step before TFT in Module B's innovation layer.

> **[7]** *A Cointegration Approach for Selection of Currency Pairs.* Indian Journal of Finance,
> December 2021. https://www.indianjournaloffinance.co.in/index.php/JF/article/view/160018/0
> Applies the distance approach and Engle-Granger two-step cointegration test across six
> currency pairs (EUR/USD, GBP/USD, USD/CAD, USD/INR, USD/JPY, USD/NZD) over multiple
> time windows (2, 5 and 10 years), finding that cointegration relationships in FX are
> time-varying and pair-dependent. Directly motivates our application of Engle-Granger
> and Johansen cointegration tests across Module B's four pairs at multiple timeframes
> in notebook 03.

> **[10]** Wang, M., Lin, Y-H. & Mikhelson, I. (2020). *Regime-Switching Factor Investing with
> Hidden Markov Models.* Journal of Risk and Financial Management, 13(12), 311. MDPI.
> https://www.mdpi.com/1911-8074/13/12/311/pdf
> Applies HMM to identify distinct market regimes on S&P 500 data over 10.5 years, then
> switches factor investment models depending on the detected regime. Out-of-sample backtesting
> confirms that HMM-based regime switching delivers higher absolute returns than any individual
> model alone. Directly motivates our use of HMM as a regime filter in Module B — detecting
> bull, bear and neutral states before the agent selects which forecasting model to trust.

> **[33a]** Guyard, K.C. & Deriaz, M. (2024). *Predicting Foreign Exchange EURUSD Direction Using
> Machine Learning.* 7th International Conference on Machine Learning and Machine Intelligence
> (MLMI 2024), Osaka, Japan. ACM.
> https://www.researchgate.net/publication/386364259_Predicting_Foreign_Exchange_EURUSD_Direction_Using_Machine_Learning
> Applies Bayesian hyperparameter search for directional forecasting of EUR/USD, achieving
> 58.52% directional accuracy and 32.48% annual return. Directly motivates our use of
> Bayesian search for tuning RF and XGBoost on FX directional classification.

> **[33b]** Öztürk, C. (2025). *Bayesian-Optimized Ensemble Learning for Multi-Class Trading Signal
> Classification.* Journal of Administrative Sciences, 24(59), 271–298.
> DOI: https://doi.org/10.35408/comuybd.1667062
> Compares RF, LightGBM, XGBoost and AdaBoost with Bayesian hyperparameter optimization,
> showing XGBoost achieves superior classification performance (Accuracy 0.974) and +49.1%
> trading return versus AdaBoost's −11.3%. Motivates our choice of XGBoost as the primary
> ensemble model in Module B's baseline stack.

> **[36]** Deep, G., Deep, A. & Lamptey, W. (2025). *Interpretable Hypothesis-Driven Trading:
> A Rigorous Walk-Forward Validation Framework for Market Microstructure Signals.*
> Texas Tech University, Department of Mathematics & Statistics. December 2025.
> Develops a rigorous walk-forward validation framework combining rolling window validation
> across 34 independent out-of-sample test periods with strict information set discipline
> to prevent lookahead bias and overfitting. Directly motivates our walk-forward validation
> protocol in validate.py with purged and embargoed splits across the same 34 OOS periods.

> **[44]** Islam, T. & Kapinchev, K. (2024). *Evaluation of the Effectiveness of Technical
> Indicators.* 17th International Conference on Signal Processing and Communication Systems
> (ICSPCS), Surfers Paradise, Australia. IEEE.
> DOI: https://doi.org/10.1109/ICSPCS63175.2024.10815753
> https://ieeexplore.ieee.org/document/10815753
> Evaluates the effectiveness of combined technical indicators — including Moving Average,
> RSI and Bollinger Bands — for identifying buy and sell areas in FX markets, finding that
> indicator combinations improve confidence over single indicators. Directly motivates our
> multi-indicator feature set combining EMA, RSI, Bollinger Bands, MACD and ATR in Module B.

---

### 1.3 Agent Architecture Declaration

This agent implements a **true reasoning agent** (not a sequential pipeline). The distinction matters:

| Property | Sequential pipeline | This agent (ReAct) |
|---|---|---|
| Control flow | Fixed script order | LLM decides which tool to call |
| Regime awareness | None | HMM-filtered market state conditions model selection |
| Explainability | Post-hoc only | Inline reasoning trace at each decision step |
| Output | Raw prediction | Structured signal with confidence + rationale |

The **backbone LLM** is `Qwen2.5-Coder-7B-Instruct` in **GGUF format** (Q4_K_M quantisation),
loaded via `llama-cpp-python`. No GPU required; peak RAM ≈ 5.2 GB.

**GGUF explained**: GGUF is the successor to GGML — a binary format for storing quantised model
weights. Using 4-bit quantisation, it reduces the 7B parameter model from ~14 GB (float16) to
~4.5 GB, enabling full CPU inference with no GPU, no API calls, and no data leaving the machine.

**ReAct framework** (Yao et al., 2022): the agent alternates between *Reason* steps (chain-of-thought
about the current market regime) and *Act* steps (calling a specific forecasting model as a tool).
The LLM receives the HMM regime label, recent OHLCV summary, and available model outputs, then
produces a structured JSON signal.

> Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2022).
> *ReAct: Synergizing Reasoning and Acting in Language Models.* arXiv:2210.03629.
> https://arxiv.org/abs/2210.03629

In [3]:
# ── Environment & imports ──────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# Statistical tests
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen

# ML
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from skopt import BayesSearchCV

# Deep learning
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# XAI
import shap

# HMM
from hmmlearn import hmm

# Local LLM backbone
from llama_cpp import Llama

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path("../src_technical")
MODEL_DIR  = Path("../models/trained")
GGUF_PATH  = Path("../models/backbone/qwen2.5-coder-7b-instruct-q4_k_m.gguf")

PAIRS      = ["EURUSDm", "GBPUSDm", "USDJPYm", "USDCHFm"]
TIMEFRAMES = ["D1", "H4", "H1"]

# ── File loader ────────────────────────────────────────────────────────────
def find_parquet(pair: str, tf: str) -> Path:
    """
    Scans DATA_DIR for a parquet file matching the pair and timeframe.
    Handles the naming pattern: ohlcv_{pair}_{tf}_{start}_{end}.parquet
    """
    matches = list(DATA_DIR.glob(f"ohlcv_{pair}_{tf}_*.parquet"))
    if len(matches) == 0:
        raise FileNotFoundError(f"No parquet found for {pair} {tf} in {DATA_DIR}")
    if len(matches) > 1:
        raise ValueError(f"Multiple parquets found for {pair} {tf}: {matches}")
    return matches[0]

# ── Verification ───────────────────────────────────────────────────────────
print("✓ Imports OK")
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.cuda.is_available()} (not required — CPU-only run)")
print(f"  XGBoost  : {xgb.__version__}")
print()
print("✓ Verifying parquet files:")
for pair in PAIRS:
    for tf in TIMEFRAMES:
        try:
            path = find_parquet(pair, tf)
            print(f"  Found: {path.name}")
        except FileNotFoundError as e:
            print(f"  MISSING: {pair} {tf} — {e}")

✓ Imports OK
  PyTorch  : 2.10.0+cpu
  CUDA     : False (not required — CPU-only run)
  XGBoost  : 3.2.0

✓ Verifying parquet files:
  Found: ohlcv_EURUSDm_D1_2021-01-03_2025-12-30.parquet
  Found: ohlcv_EURUSDm_H4_2021-01-03_2025-12-30.parquet
  Found: ohlcv_EURUSDm_H1_2021-01-03_2025-12-30.parquet
  Found: ohlcv_GBPUSDm_D1_2021-01-03_2025-12-30.parquet
  Found: ohlcv_GBPUSDm_H4_2021-01-03_2025-12-30.parquet
  MISSING: GBPUSDm H1 — No parquet found for GBPUSDm H1 in ..\src_technical
  Found: ohlcv_USDJPYm_D1_2021-01-03_2025-12-30.parquet
  Found: ohlcv_USDJPYm_H4_2021-01-03_2025-12-30.parquet
  Found: ohlcv_USDJPYm_H1_2021-01-03_2025-12-30.parquet
  Found: ohlcv_USDCHFm_D1_2021-01-03_2025-12-30.parquet
  Found: ohlcv_USDCHFm_H4_2021-01-03_2025-12-30.parquet
  Found: ohlcv_USDCHFm_H1_2021-01-03_2025-12-30.parquet


## 2. Data Loading — Architecture Rationale

Before any code: why these indicators, and why these timeframes?

### Timeframe hierarchy

| Timeframe | Role in this agent |
|---|---|
| **D1** | Trend direction — EMA 200 regime, weekly momentum |
| **H4** | Swing structure — key S/R levels, MACD crossovers |
| **H1** | Entry-level signal — RSI extremes, ATR-normalised volatility |

Each pair is loaded at all three timeframes. The agent's ReAct loop can query any combination.

### Why each indicator

| Indicator | What it captures | Paper justification |
|---|---|---|
| RSI (14) | Mean-reversion pressure; overbought/oversold extremes | [TA], [33b] |
| MACD (12/26/9) | Trend momentum; divergence from price | [33b] |
| Bollinger Bands (20, 2σ) | Volatility regime; %B for squeeze detection | [TA] |
| ATR (14) | Normalised volatility; position sizing input | Wilder (1978) |
| EMA 20/50/200 | Trend hierarchy; golden/death cross signals | [TA] |
| Log returns | Stationarity; required for ADF test to pass | Hamilton (1994) |
| Momentum (n-period) | Serial autocorrelation; directional persistence | [33b] |

Log returns are used instead of raw prices because raw OHLCV prices are I(1) processes
(unit root present). This is a foundational result in financial econometrics established
by Hamilton (1994), confirmed empirically on our own data through ADF testing in the
cell below.

> Hamilton, J.D. (1994). *Time Series Analysis.* Princeton University Press, Princeton, NJ.
> ISBN: 978-0-691-04289-3

In [9]:
def load_pair(pair: str, tf: str) -> pd.DataFrame:
    """
    Load a single OHLCV parquet using glob pattern matching.
    Returns DataFrame with DatetimeIndex, lowercase columns.
    """
    path = find_parquet(pair, tf)
    df = pd.read_parquet(path, engine="pyarrow")
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df.columns = df.columns.str.lower()
    assert {"open","high","low","close","volume"}.issubset(df.columns), \
        f"Missing OHLCV columns in {path.name}"
    return df


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute all technical indicators. Returns copy with new columns."""
    df = df.copy()
    c = df["close"]

    # ── Log returns (Hamilton 1994) ───────────────────────────────────────
    df["log_ret"]    = np.log(c / c.shift(1))
    df["log_ret_sq"] = df["log_ret"] ** 2

    # ── RSI (14) — Islam & Kapinchev 2024, Öztürk 2025 ───────────────────
    delta = c.diff()
    gain  = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(span=14, adjust=False).mean()
    df["rsi"] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))

    # ── MACD (12, 26, 9) — Öztürk 2025 ──────────────────────────────────
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    df["macd"]        = ema12 - ema26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_hist"]   = df["macd"] - df["macd_signal"]

    # ── Bollinger Bands (20, 2σ) — Islam & Kapinchev 2024 ────────────────
    sma20 = c.rolling(20).mean()
    std20 = c.rolling(20).std()
    df["bb_upper"] = sma20 + 2 * std20
    df["bb_lower"] = sma20 - 2 * std20
    df["bb_pct"]   = (c - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])

    # ── ATR (14) — Wilder 1978 ────────────────────────────────────────────
    hl  = df["high"] - df["low"]
    hpc = (df["high"] - c.shift(1)).abs()
    lpc = (df["low"]  - c.shift(1)).abs()
    tr  = pd.concat([hl, hpc, lpc], axis=1).max(axis=1)
    df["atr"] = tr.ewm(span=14, adjust=False).mean()

    # ── EMAs — Islam & Kapinchev 2024 ────────────────────────────────────
    for span in [20, 50, 200]:
        df[f"ema_{span}"] = c.ewm(span=span, adjust=False).mean()

    # ── Momentum — Öztürk 2025 ───────────────────────────────────────────
    df["mom_5"]  = c.pct_change(5)
    df["mom_20"] = c.pct_change(20)

    # ── Target (next-bar direction) ───────────────────────────────────────
    df["target"] = (df["log_ret"].shift(-1) > 0).astype(int)

    return df.dropna()


def adf_test(series: pd.Series, name: str) -> dict:
    """Augmented Dickey-Fuller test. Returns stat, p-value, verdict."""
    stat, p, _, _, crit, _ = adfuller(series.dropna(), autolag="AIC")
    result = {"series": name, "stat": round(stat, 4),
              "p_value": round(p, 4), "stationary": p < 0.05}
    print(f"  ADF {name:<28} stat={stat:7.3f}  p={p:.4f}  "
          f"{'✓ stationary' if result['stationary'] else '✗ unit root'}")
    return result

# ── Load all available pairs / timeframes ─────────────────────────────────
MISSING = set()  # known missing — awaiting data from team

data = {}
adf_results = []
skipped = []

for pair in PAIRS:
    data[pair] = {}
    for tf in TIMEFRAMES:
        if (pair, tf) in MISSING:
            print(f"  SKIPPED: {pair} {tf} — file not yet available")
            skipped.append(f"{pair}_{tf}")
            continue
        df = load_pair(pair, tf)
        df = add_features(df)
        data[pair][tf] = df
        r = adf_test(df["log_ret"], f"{pair}_{tf}_logret")
        adf_results.append(r)

adf_df = pd.DataFrame(adf_results)

print(f"\n✓ Loaded {len(adf_results)} datasets successfully")
print(f"  Skipped : {skipped}")
print(f"  Stationarity: {adf_df['stationary'].sum()}/{len(adf_df)} log return series pass ADF at p<0.05")
print(f"  (All should pass — raw price series would fail, as expected)")

  ADF EURUSDm_D1_logret            stat=-40.079  p=0.0000  ✓ stationary
  ADF EURUSDm_H4_logret            stat=-22.274  p=0.0000  ✓ stationary
  ADF EURUSDm_H1_logret            stat=-25.869  p=0.0000  ✓ stationary
  ADF GBPUSDm_D1_logret            stat=-29.193  p=0.0000  ✓ stationary
  ADF GBPUSDm_H4_logret            stat=-21.962  p=0.0000  ✓ stationary
  ADF GBPUSDm_H1_logret            stat=-31.162  p=0.0000  ✓ stationary
  ADF USDJPYm_D1_logret            stat=-40.046  p=0.0000  ✓ stationary
  ADF USDJPYm_H4_logret            stat=-62.777  p=0.0000  ✓ stationary
  ADF USDJPYm_H1_logret            stat=-35.515  p=0.0000  ✓ stationary
  ADF USDCHFm_D1_logret            stat=-39.579  p=0.0000  ✓ stationary
  ADF USDCHFm_H4_logret            stat=-52.013  p=0.0000  ✓ stationary
  ADF USDCHFm_H1_logret            stat=-31.172  p=0.0000  ✓ stationary

✓ Loaded 12 datasets successfully
  Skipped : []
  Stationarity: 12/12 log return series pass ADF at p<0.05
  (All should pass — raw pr

In [10]:
# ── ADF on raw close prices (should fail — I(1) expected) ─────────────────
print("ADF on raw close prices (unit root expected — should NOT be stationary):\n")

close_results = []
for pair in PAIRS:
    for tf in TIMEFRAMES:
        if pair not in data or tf not in data[pair]:
            continue
        series = data[pair][tf]["close"].dropna()
        stat, p, lags, nobs, crit = adfuller(series, maxlag=5, autolag=None)
        result = {"series": f"{pair}_{tf}_close", "stat": round(stat, 4),
                  "p_value": round(p, 4), "stationary": p < 0.05}
        close_results.append(result)
        label = "✓ stationary" if p < 0.05 else "✗ unit root"
        print(f"  ADF {pair}_{tf}_close".ljust(36) + f"stat={stat:7.3f}  p={p:.4f}  {label}")

close_df = pd.DataFrame(close_results)
print(f"\n✓ Raw close stationarity: {close_df['stationary'].sum()}/{len(close_df)} pass ADF")
print(f"  (Expect 0 — raw prices are I(1) processes, confirming log return transformation is necessary)")

ADF on raw close prices (unit root expected — should NOT be stationary):

  ADF EURUSDm_D1_close              stat= -1.829  p=0.3660  ✗ unit root
  ADF EURUSDm_H4_close              stat= -2.062  p=0.2600  ✗ unit root
  ADF EURUSDm_H1_close              stat= -2.055  p=0.2630  ✗ unit root
  ADF GBPUSDm_D1_close              stat= -1.727  p=0.4174  ✗ unit root
  ADF GBPUSDm_H4_close              stat= -1.823  p=0.3693  ✗ unit root
  ADF GBPUSDm_H1_close              stat= -1.834  p=0.3639  ✗ unit root
  ADF USDJPYm_D1_close              stat= -1.619  p=0.4730  ✗ unit root
  ADF USDJPYm_H4_close              stat= -1.624  p=0.4704  ✗ unit root
  ADF USDJPYm_H1_close              stat= -1.653  p=0.4557  ✗ unit root
  ADF USDCHFm_D1_close              stat= -0.994  p=0.7553  ✗ unit root
  ADF USDCHFm_H4_close              stat= -0.980  p=0.7604  ✗ unit root
  ADF USDCHFm_H1_close              stat= -1.000  p=0.7533  ✗ unit root

✓ Raw close stationarity: 0/12 pass ADF
  (Expect 0 — raw pri

### ADF Test Results — Interpretation

Two rounds of ADF testing were conducted to validate the log return transformation.

**Raw close prices (I(1) — non-stationary):** All 12 series fail the ADF test
with p-values ranging from 0.26 to 0.76, confirming the presence of a unit root
across all pairs and timeframes. Raw prices cannot be used directly as model inputs.

**Log returns (I(0) — stationary):** All 12 log return series reject the unit root
hypothesis at p ≈ 0.000, with ADF statistics ranging from -21.9 to -62.8 —
well beyond the 1% critical threshold. Log returns are confirmed as the correct
stationary transformation following Hamilton (1994).

This two-step verification is not merely theoretical — it is demonstrated empirically
on our exact dataset before any model training begins.